# 1. Initial Exploration: Market Score Direction

**Question:** Does a higher reference score represent a stronger housing market?

This notebook maps each valid reference score to the corresponding ZIP's latest available 2025 SoCal record, then evaluates five pieces of evidence:

1. Reference coverage and price-level relationships
2. Rank relationships with stronger-market signals
3. Comparison of low-, middle-, and high-score groups
4. Size-normalized supply and demand measures
5. Multivariate Ridge regression as a diagnostic only

The analysis treats `Zip Code` as an identifier, not a predictor. A reference score is assigned only to the latest 2025 row for its ZIP, never to every historical row.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
sns.set_theme(style="whitegrid", context="notebook")

# Works when run from either the repository root or the notebooks folder.
ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

SOCAL_PATH = ROOT / "data" / "SoCal.csv"
REFERENCE_PATH = ROOT / "data" / "market_score_reference.csv"

def normalize_zip(series):
    # Return a five-character ZIP string without converting missing values to 'nan'.
    cleaned = series.astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
    return cleaned.str.zfill(5)

def corr_pair(frame, x, y="Market_Score"):
    pair = frame[[x, y]].dropna()
    return pd.Series({
        "n": len(pair),
        "Pearson": pair[x].corr(pair[y], method="pearson"),
        "Spearman": pair[x].corr(pair[y], method="spearman"),
    })

## Load and prepare the data

Important rules:

- Parse ZIP codes as text.
- Convert nonnumeric reference entries such as `/` to missing.
- Select the maximum `PERIOD_END` **within 2025 separately for each ZIP**.
- Map one numeric score per ZIP only after checking for duplicate numeric references.

In [ ]:
socal = pd.read_csv(SOCAL_PATH, dtype={"Zip Code": "string"})
reference_raw = pd.read_csv(REFERENCE_PATH, dtype={"Zip Code": "string"})

socal["Zip Code"] = normalize_zip(socal["Zip Code"])
reference_raw["Zip Code"] = normalize_zip(reference_raw["Zip Code"])
socal["PERIOD_BEGIN"] = pd.to_datetime(socal["PERIOD_BEGIN"], errors="coerce")
socal["PERIOD_END"] = pd.to_datetime(socal["PERIOD_END"], errors="coerce")
reference_raw["Market_Score"] = pd.to_numeric(reference_raw["Market Score"], errors="coerce")

# Only numeric score rows are eligible for calibration. The raw reference file can
# contain repeated ZIP rows with placeholders, so uniqueness is tested after this filter.
reference = reference_raw.dropna(subset=["Market_Score"]).copy()
numeric_duplicate_count = int(reference["Zip Code"].duplicated().sum())
if numeric_duplicate_count:
    conflicts = reference.groupby("Zip Code")["Market_Score"].nunique()
    if conflicts.gt(1).any():
        raise ValueError("A ZIP has conflicting numeric reference scores.")
    reference = reference.drop_duplicates("Zip Code", keep="first")

socal_2025 = socal.loc[socal["PERIOD_END"].dt.year.eq(2025)].copy()
latest_2025 = (
    socal_2025.sort_values(["Zip Code", "PERIOD_END"])
    .groupby("Zip Code", as_index=False, group_keys=False)
    .tail(1)
    .copy()
)

score_map = reference.set_index("Zip Code")["Market_Score"]
latest_2025["Market_Score"] = latest_2025["Zip Code"].map(score_map)
analysis = latest_2025.dropna(subset=["Market_Score"]).copy()

audit = pd.DataFrame({
    "Check": [
        "SoCal rows", "Unique SoCal ZIPs", "ZIPs with a 2025 record",
        "ZIPs without a 2025 record", "Raw reference rows",
        "Numeric reference scores", "Nonnumeric/blank reference entries",
        "Duplicate ZIPs among numeric references", "Matched analysis ZIPs",
        "Duplicate ZIPs in latest-2025 table"
    ],
    "Count": [
        len(socal), socal["Zip Code"].nunique(), latest_2025["Zip Code"].nunique(),
        socal["Zip Code"].nunique() - latest_2025["Zip Code"].nunique(),
        len(reference_raw), len(reference), reference_raw["Market_Score"].isna().sum(),
        numeric_duplicate_count, len(analysis), latest_2025["Zip Code"].duplicated().sum()
    ]
})
display(audit.style.hide(axis="index").format({"Count": "{:,.0f}"}))

assert analysis["Zip Code"].is_unique
assert analysis["PERIOD_END"].dt.year.eq(2025).all()
assert analysis["Market_Score"].between(0, 100).all()

display(Markdown(
    f"**Prepared sample:** {len(analysis):,} ZIPs have both a valid numeric score and "
    f"a latest-2025 market record. Latest dates range from "
    f"{analysis['PERIOD_END'].min():%Y-%m-%d} to {analysis['PERIOD_END'].max():%Y-%m-%d}."
))

## Analysis 1 — Price is not the same as market strength

Test whether score is simply a proxy for how expensive a ZIP is. Both Pearson correlation (linear association) and Spearman correlation (rank association) are shown.

In [ ]:
price_metrics = ["MEDIAN_SALE_PRICE", "MEDIAN_LIST_PRICE", "MEDIAN_PPSF"]
price_labels = {
    "MEDIAN_SALE_PRICE": "Median sale price",
    "MEDIAN_LIST_PRICE": "Median list price",
    "MEDIAN_PPSF": "Median sale price per sq. ft.",
}

price_corr = pd.DataFrame({m: corr_pair(analysis, m) for m in price_metrics}).T
price_corr.index = [price_labels[m] for m in price_metrics]
display(price_corr.style.format({"n": "{:.0f}", "Pearson": "{:.3f}", "Spearman": "{:.3f}"}))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, metric in zip(axes, price_metrics):
    sns.regplot(data=analysis, x=metric, y="Market_Score", ax=ax,
                scatter_kws={"alpha": 0.65, "s": 35}, line_kws={"color": "#C44E52"})
    ax.set_title(price_labels[metric])
    ax.set_xlabel(price_labels[metric])
    ax.set_ylabel("Reference market score")
plt.suptitle("Reference score versus price levels", y=1.03, fontsize=14)
plt.tight_layout()
plt.show()

max_abs_price_rho = price_corr["Spearman"].abs().max()
display(Markdown(
    f"**Conclusion:** The largest absolute price-level Spearman correlation is "
    f"**{max_abs_price_rho:.2f}**. Price level alone does not define the score; "
    "an expensive ZIP can still have a slow or weak current market."
))

## Analysis 2 — Rank relationships with stronger-market signals

Spearman correlation is the primary measure because the score may be ordinal or nonlinear. The `Expected direction` column encodes housing-market intuition. For days on market and inventory, lower values normally indicate a stronger seller's market.

In [ ]:
signal_direction = {
    "AVG_SALE_TO_LIST": 1,
    "SOLD_ABOVE_LIST": 1,
    "OFF_MARKET_IN_TWO_WEEKS": 1,
    "MEDIAN_DOM": -1,
    "INVENTORY": -1,
    "HOMES_SOLD": 1,
    "PENDING_SALES": 1,
    "NEW_LISTINGS": 1,
}
signal_labels = {
    "AVG_SALE_TO_LIST": "Average sale-to-list ratio",
    "SOLD_ABOVE_LIST": "Share sold above list",
    "OFF_MARKET_IN_TWO_WEEKS": "Share off market within 2 weeks",
    "MEDIAN_DOM": "Median days on market",
    "INVENTORY": "Inventory",
    "HOMES_SOLD": "Homes sold",
    "PENDING_SALES": "Pending sales",
    "NEW_LISTINGS": "New listings",
}

rows = []
for metric, direction in signal_direction.items():
    stats = corr_pair(analysis, metric)
    rows.append({
        "Metric": signal_labels[metric],
        "Expected direction": "+" if direction > 0 else "−",
        "n": stats["n"],
        "Spearman": stats["Spearman"],
        "Direction-adjusted correlation": stats["Spearman"] * direction,
    })
signal_corr = pd.DataFrame(rows).sort_values("Direction-adjusted correlation", ascending=False)
display(signal_corr.style.hide(axis="index").format({
    "n": "{:.0f}", "Spearman": "{:.3f}", "Direction-adjusted correlation": "{:.3f}"
}))

plt.figure(figsize=(9, 4.8))
sns.barplot(data=signal_corr, y="Metric", x="Spearman", color="#4C72B0")
plt.axvline(0, color="black", linewidth=0.8)
plt.xlim(-1, 1)
plt.title("Rank correlation between market signals and reference score")
plt.xlabel("Spearman correlation")
plt.ylabel("")
plt.tight_layout()
plt.show()

direction_consistency = (signal_corr["Direction-adjusted correlation"] > 0).mean()
display(Markdown(
    f"**Conclusion:** {direction_consistency:.0%} of tested signals move in the economically expected "
    "direction. The strongest evidence should come from speed, liquidity, and competition measures rather "
    "than price levels."
))

## Analysis 3 — Compare low-, middle-, and high-score groups

The reference sample is divided into three approximately equal score groups. Group medians reveal monotonic patterns that a simple linear correlation can miss.

In [ ]:
analysis["Score group"] = pd.qcut(
    analysis["Market_Score"], q=3, labels=["Low", "Middle", "High"], duplicates="drop"
)
group_order = ["Low", "Middle", "High"]
group_metrics = ["MEDIAN_DOM", "AVG_SALE_TO_LIST", "SOLD_ABOVE_LIST", "OFF_MARKET_IN_TWO_WEEKS", "INVENTORY"]

group_medians = analysis.groupby("Score group", observed=True)[group_metrics].median().reindex(group_order)
group_medians.columns = [signal_labels[c] for c in group_metrics]
display(group_medians.style.format("{:.3f}").background_gradient(cmap="Blues", axis=0))

# Standardize each metric across the three group medians and reverse weak-market measures.
heat = group_medians.copy()
for column in heat.columns:
    std = heat[column].std(ddof=0)
    heat[column] = 0 if std == 0 else (heat[column] - heat[column].mean()) / std
for column in ["Median days on market", "Inventory"]:
    heat[column] *= -1

plt.figure(figsize=(10, 3.2))
sns.heatmap(heat.T, annot=True, fmt=".2f", cmap="RdYlBu", center=0, cbar_kws={"label": "Strength-adjusted z-score"})
plt.title("Market signals by reference-score group")
plt.xlabel("Reference-score group")
plt.ylabel("")
plt.tight_layout()
plt.show()

display(Markdown(
    "**Conclusion:** If the high-score column is generally stronger (bluer) than the low-score column, "
    "the ordering supports **100 = stronger market**, even when the relationship is not perfectly linear."
))

## Analysis 4 — Use size-normalized supply and demand measures

Raw counts reflect ZIP size as well as market conditions. Ratios can better capture market balance. These are exploratory proxies, not official source metrics.

In [ ]:
analysis["INVENTORY_PER_HOME_SOLD"] = analysis["INVENTORY"] / analysis["HOMES_SOLD"].replace(0, np.nan)
analysis["PENDING_PER_NEW_LISTING"] = analysis["PENDING_SALES"] / analysis["NEW_LISTINGS"].replace(0, np.nan)
analysis["HOMES_SOLD_PER_NEW_LISTING"] = analysis["HOMES_SOLD"] / analysis["NEW_LISTINGS"].replace(0, np.nan)

derived_direction = {
    "INVENTORY_PER_HOME_SOLD": -1,
    "PENDING_PER_NEW_LISTING": 1,
    "HOMES_SOLD_PER_NEW_LISTING": 1,
}
derived_labels = {
    "INVENTORY_PER_HOME_SOLD": "Inventory per home sold",
    "PENDING_PER_NEW_LISTING": "Pending sales per new listing",
    "HOMES_SOLD_PER_NEW_LISTING": "Homes sold per new listing",
}

derived_rows = []
for metric, direction in derived_direction.items():
    stats = corr_pair(analysis, metric)
    derived_rows.append({
        "Metric": derived_labels[metric], "Expected direction": "+" if direction > 0 else "−",
        "n": stats["n"], "Spearman": stats["Spearman"],
        "Direction-adjusted correlation": stats["Spearman"] * direction,
    })
derived_corr = pd.DataFrame(derived_rows).sort_values("Direction-adjusted correlation", ascending=False)
display(derived_corr.style.hide(axis="index").format({
    "n": "{:.0f}", "Spearman": "{:.3f}", "Direction-adjusted correlation": "{:.3f}"
}))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, metric in zip(axes, derived_direction):
    sns.regplot(data=analysis, x=metric, y="Market_Score", ax=ax,
                scatter_kws={"alpha": 0.65, "s": 35}, line_kws={"color": "#C44E52"})
    ax.set_title(derived_labels[metric])
    ax.set_xlabel(derived_labels[metric])
    ax.set_ylabel("Reference market score")
plt.suptitle("Reference score versus size-normalized market balance", y=1.03, fontsize=14)
plt.tight_layout()
plt.show()

best_derived = derived_corr.iloc[0]
display(Markdown(
    f"**Conclusion:** The strongest derived indicator is **{best_derived['Metric']}**, with a "
    f"direction-adjusted Spearman correlation of **{best_derived['Direction-adjusted correlation']:.2f}**. "
    "Ratios should complement, not automatically replace, their component variables."
))

## Analysis 5 — Multivariate diagnostic

Individual relationships may be weak because the reference score combines several market dimensions. A Ridge regression is therefore used **only as a diagnostic**: can interpretable signals collectively recover the reference ordering?

The model excludes identifiers and price levels. It uses nested five-fold cross-validation: the inner folds choose Ridge strength and the outer folds estimate out-of-sample performance.

In [ ]:
model_features = [
    "MEDIAN_DOM", "AVG_SALE_TO_LIST", "SOLD_ABOVE_LIST", "OFF_MARKET_IN_TWO_WEEKS",
    "INVENTORY_PER_HOME_SOLD", "PENDING_PER_NEW_LISTING", "HOMES_SOLD_PER_NEW_LISTING",
]
X = analysis[model_features]
y = analysis["Market_Score"]

base_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
    ("ridge", Ridge())
])
alpha_grid = np.logspace(-3, 3, 25)
inner_cv = KFold(n_splits=5, shuffle=True, random_state=17)
outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)
search = GridSearchCV(base_pipeline, {"ridge__alpha": alpha_grid}, scoring="neg_mean_absolute_error", cv=inner_cv)
cv_pred = cross_val_predict(search, X, y, cv=outer_cv)

performance = pd.DataFrame({
    "Metric": ["Cross-validated MAE", "Cross-validated RMSE", "Cross-validated R²", "Predicted vs actual Spearman"],
    "Value": [
        mean_absolute_error(y, cv_pred),
        mean_squared_error(y, cv_pred) ** 0.5,
        r2_score(y, cv_pred),
        pd.Series(cv_pred).corr(y.reset_index(drop=True), method="spearman"),
    ]
})
display(performance.style.hide(axis="index").format({"Value": "{:.3f}"}))

plt.figure(figsize=(6, 5))
sns.scatterplot(x=y, y=cv_pred, s=55, alpha=0.75)
lims = [min(y.min(), cv_pred.min()), max(y.max(), cv_pred.max())]
plt.plot(lims, lims, "--", color="#C44E52", label="Perfect agreement")
plt.xlim(lims); plt.ylim(lims)
plt.xlabel("Actual reference score")
plt.ylabel("Cross-validated predicted score")
plt.title("Out-of-sample multivariate diagnostic")
plt.legend()
plt.tight_layout()
plt.show()

# Final-sample coefficients are for interpretation, not unbiased performance measurement.
final_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("ridge", RidgeCV(alphas=alpha_grid, cv=5, scoring="neg_mean_absolute_error"))
])
final_pipe.fit(X, y)
coef = pd.DataFrame({
    "Metric": [derived_labels.get(c, signal_labels.get(c, c)) for c in model_features],
    "Standardized coefficient": final_pipe.named_steps["ridge"].coef_,
}).sort_values("Standardized coefficient")
display(coef.style.hide(axis="index").format({"Standardized coefficient": "{:.3f}"}))

plt.figure(figsize=(9, 4.5))
sns.barplot(data=coef, y="Metric", x="Standardized coefficient", color="#55A868")
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Ridge coefficients fitted to the full matched sample")
plt.xlabel("Change in score per one-standard-deviation increase")
plt.ylabel("")
plt.tight_layout()
plt.show()

cv_rho = performance.loc[performance["Metric"].eq("Predicted vs actual Spearman"), "Value"].iloc[0]
display(Markdown(
    f"**Conclusion:** The out-of-sample predicted-versus-actual rank correlation is **{cv_rho:.2f}**. "
    "A positive and material value supports the interpretation that the score combines several market-strength "
    "signals. This diagnostic does not yet approve a production scoring model."
))

## Overall conclusion and reminders

Use the combined evidence, not one price correlation:

- A high score is consistent with a stronger market if competition and speed metrics move in the expected direction, high-score groups look stronger, and the multivariate diagnostic preserves score ordering.
- Weak relationships with sale or list price are not contradictory. Price measures market level; the reference score appears intended to measure market conditions.
- The reference file has no timestamp. This notebook follows the assignment's explicit assumption that each score belongs to its ZIP's latest available 2025 row.
- Latest dates can differ by ZIP. Keep `PERIOD_END` in evaluation outputs and consider date-mix sensitivity.
- Rolling 90-day observations overlap, so historical rows are not independent monthly samples.
- Do not copy the latest reference score onto earlier periods.
- Before final modeling, review missingness, outliers, geographic representativeness, temporal stability, and whether coefficient signs remain economically credible.